In [34]:

import os
import geopandas as gpd
import fiona
import pandas as pd

### Function to take out columns with no data, data less than 25% filled, and specific named columns

In [35]:
def clean_csv(input_path, output_path, threshold=0.25, drop_cols=None):
    df = pd.read_csv(input_path, dtype=str).replace(r"^\s*$", pd.NA, regex=True)

    sparse  = df.columns[df.notna().mean() < threshold].tolist()
    named   = [c for c in (drop_cols or []) if c in df.columns]
    missing = [c for c in (drop_cols or []) if c not in df.columns]

    df.drop(columns=list(dict.fromkeys(sparse + named)), inplace=True)
    df.to_csv(output_path, index=False)

    print(f"\n{input_path} → {output_path} | {len(df.columns)} columns kept")
    print(f"  Dropped sparse : {sparse or 'none'}")
    print(f"  Dropped named  : {named or 'none'}")
    
    return df

#### Calling Cleaning Function

In [36]:
clean_capacity = clean_csv("data/capacity.csv", "data/capacity_new.csv", threshold = 0.25, drop_cols = ["source_id","comment"])
clean_coal = clean_csv("data/coal.csv", "data/coal_new.csv", threshold = 0.25, drop_cols = ["source_id","comment", "amount_sold_tonnes"])
clean_commodities = clean_csv("data/commodities.csv", "data/commodoties_new.csv", threshold = 0.25, drop_cols = ["source_id","comment", "amount_sold_tonnes","metal_payable_tonnes","mine_processing"])
clean_minerals = clean_csv("data/minerals.csv", "data/minerals_new.csv", threshold = 0.25, drop_cols = ["source_id","comment", "amount_sold_tonnes","mine_processing"])
clean_reserves = clean_csv("data/reserves.csv", "data/reserves_new.csv", threshold = 0.25, drop_cols = ["source_id","comment"])
clean_waste = clean_csv("data/waste.csv", "data/waste_new.csv", threshold = 0.25, drop_cols = ["source_id","comment"])


data/capacity.csv → data/capacity_new.csv | 5 columns kept
  Dropped sparse : ['material', 'comment']
  Dropped named  : ['source_id', 'comment']

data/coal.csv → data/coal_new.csv | 6 columns kept
  Dropped sparse : ['amount_sold_tonnes', 'reporting_period', 'comment']
  Dropped named  : ['source_id', 'comment', 'amount_sold_tonnes']

data/commodities.csv → data/commodoties_new.csv | 9 columns kept
  Dropped sparse : ['yield_ppm', 'amount_sold_tonnes', 'metal_payable_tonnes', 'mine_processing', 'reporting_period', 'comment']
  Dropped named  : ['source_id', 'comment', 'amount_sold_tonnes', 'metal_payable_tonnes', 'mine_processing']

data/minerals.csv → data/minerals_new.csv | 6 columns kept
  Dropped sparse : ['overall_grade_ppm', 'amount_sold_tonnes', 'mine_processing', 'reporting_period', 'comment']
  Dropped named  : ['source_id', 'comment', 'amount_sold_tonnes', 'mine_processing']

data/reserves.csv → data/reserves_new.csv | 8 columns kept
  Dropped sparse : ['comment']
  Dropped

### Changing strings to be numbers so we can do calculations (i.e. reserve back/forward filling)

In [37]:
# reserves
clean_reserves["mineral_value_tonnes"]   = pd.to_numeric(clean_reserves["mineral_value_tonnes"], errors="coerce")
clean_reserves["commodity_value_tonnes"] = pd.to_numeric(clean_reserves["commodity_value_tonnes"], errors="coerce")
clean_reserves["grade_ppm"]              = pd.to_numeric(clean_reserves["grade_ppm"], errors="coerce")

# coal
clean_coal["value_tonnes"]               = pd.to_numeric(clean_coal["value_tonnes"], errors="coerce")

# minerals
clean_minerals["value_tonnes"]           = pd.to_numeric(clean_minerals["value_tonnes"], errors="coerce")

# commodities
clean_commodities["value_tonnes"]        = pd.to_numeric(clean_commodities["value_tonnes"], errors="coerce")
clean_commodities["grade_ppm"]           = pd.to_numeric(clean_commodities["grade_ppm"], errors="coerce")
clean_commodities["recovery_rate"]       = pd.to_numeric(clean_commodities["recovery_rate"], errors="coerce")

# waste
clean_waste["value_tonnes"]              = pd.to_numeric(clean_waste["value_tonnes"], errors="coerce")
clean_waste["total_material_tonnes"]     = pd.to_numeric(clean_waste["total_material_tonnes"], errors="coerce")

# capacity
clean_capacity["value_tpa"]              = pd.to_numeric(clean_capacity["value_tpa"], errors="coerce")

print("Done!")
# Want to be using filename_new after running this cell for future uses

Done!


### Function for Back/Forward Filling of Reserve Data Values

In [ ]:
output_path   = "data/reserves_filled.csv"

# Add together coal and mineral production files
production = pd.concat([
    clean_coal[clean_coal["type"] == "Coal mined"][["facility_id", "year", "material", "value_tonnes"]],
    clean_minerals[clean_minerals["type"] == "Ore mined"][["facility_id", "year", "material", "value_tonnes"]]
], ignore_index=True)

# Backfill and forward fill from reserve values given
# If 2 values given, assume new prospecting at later date, and only forward fill from this point
def fill_one_group(known_rows, production):
    fid, mat   = known_rows["facility_id"].iloc[0], known_rows["material"].iloc[0]
    anchors    = dict(zip(known_rows["year"], known_rows["mineral_value_tonnes"]))
    prod       = production[(production["facility_id"] == fid) & (production["material"] == mat)]
    prod       = prod.set_index("year")["value_tonnes"].to_dict()
    if not prod:
        return pd.DataFrame()

    anchor_years = sorted(anchors)
    estimates    = {}

    for i, ay in enumerate(anchor_years):
        av          = anchors[ay]
        next_anchor = anchor_years[i + 1] if i + 1 < len(anchor_years) else None
        if pd.isna(av):
            continue

        val = av
        for y in sorted(y for y in prod if y > ay):
            if next_anchor and y >= next_anchor:
                break
            val -= prod[y]
            estimates[y] = val

        if i == 0:
            val = av
            for y in sorted((y for y in prod if y < ay), reverse=True):
                val += prod[y]
                estimates[y] = val

    template = known_rows.iloc[0].to_dict()
    return pd.DataFrame([
        {**template, "year": y, "mineral_value_tonnes": v,
         "commodity_value_tonnes": None, "grade_ppm": None,
         "source_id": "filled", "comment": None}
        for y, v in estimates.items() if y not in anchors
    ])

#### Run Code to get new reserve values

In [39]:
# Run the code and recieve new CSV file
reserves   = clean_reserves

filled_parts = [fill_one_group(g, production)
                for _, g in reserves.groupby(["facility_id", "material"])]
filled_parts = [f for f in filled_parts if not f.empty]

all_filled = pd.concat(filled_parts, ignore_index=True) if filled_parts else pd.DataFrame()
result     = pd.concat([reserves, all_filled], ignore_index=True) if not all_filled.empty else reserves.copy()

result.sort_values(["facility_id", "material", "year"], inplace=True)
result.to_csv(output_path, index=False)

print(f"Original rows : {len(reserves)}")
print(f"Filled rows   : {len(all_filled)}")

Original rows : 1039
Filled rows   : 4476


### Code to Remove Rows of Data found by ID in ids_no_start_date

In [ ]:
#Still need to make sure that the variable id_no_start_date is in the right place for this to run

%run "data_processing_final2.ipynb"
print("this is before",len(clean_capacity))
def remove_no_start_date(df, ids_to_remove):
    return df[~df["facility_id"].isin(ids_to_remove)]

clean_capacity    = remove_no_start_date(clean_capacity, ids_no_start_date)
clean_coal        = remove_no_start_date(clean_coal, ids_no_start_date)
clean_commodities = remove_no_start_date(clean_commodities, ids_no_start_date)
clean_minerals    = remove_no_start_date(clean_minerals, ids_no_start_date)
clean_reserves    = remove_no_start_date(clean_reserves, ids_no_start_date)
clean_waste       = remove_no_start_date(clean_waste, ids_no_start_date)
print("this is after",len(cleaner_capacity))

/Users/michelenaorourke/Desktop/6.C51/1.C51/1.C51-Final-Project
Layer: facilities
Columns:
column name: facility_id data type: str
column name: facility_name data type: str
column name: facility_other_names data type: str
column name: sub_site_name data type: object
column name: sub_site_other_names data type: object
column name: facility_type data type: str
column name: primary_commodity data type: str
column name: commodities_products data type: str
column name: facility_equipment data type: str
column name: production_start data type: float64
column name: production_end data type: float64
column name: activity_status data type: str
column name: activity_status_year data type: float64
column name: surface_area_sq_km data type: float64
column name: concession_area_sq_km data type: float64
column name: country data type: str
column name: GID_0 data type: str
column name: GID_1 data type: str
column name: GID_2 data type: str
column name: GID_3 data type: str
column name: GID_4 data typ

In [41]:
#pip install nbformat
%run data_processing_final2.ipynb
print(ids_no_start_date)


/Users/michelenaorourke/Desktop/6.C51/1.C51/1.C51-Final-Project
Layer: facilities
Columns:
column name: facility_id data type: str
column name: facility_name data type: str
column name: facility_other_names data type: str
column name: sub_site_name data type: object
column name: sub_site_other_names data type: object
column name: facility_type data type: str
column name: primary_commodity data type: str
column name: commodities_products data type: str
column name: facility_equipment data type: str
column name: production_start data type: float64
column name: production_end data type: float64
column name: activity_status data type: str
column name: activity_status_year data type: float64
column name: surface_area_sq_km data type: float64
column name: concession_area_sq_km data type: float64
column name: country data type: str
column name: GID_0 data type: str
column name: GID_1 data type: str
column name: GID_2 data type: str
column name: GID_3 data type: str
column name: GID_4 data typ

## Function to merge all of the .csv files together including the facilities data and the price data

In [ ]:
# Load facilities and price data
facilities = pd.read_csv("data/facilities_data_after_dropping_start_dates_only_mines.csv")
prices = pd.read_csv("data/price_data.csv")

# Capacity - aggregate to one row per facility_id + year
capacity_pivot = clean_capacity.groupby(["facility_id", "year"]).agg({
    "value_tpa": "sum"
}).reset_index()
capacity_pivot.columns = ["facility_id", "year", "capacity_value_tpa"]

#Coal - pivot by type (clean coal vs coal mined)
coal_pivot = clean_coal.pivot_table(
    index=["facility_id", "year"],
    columns="type",
    values="value_tonnes",
    aggfunc="sum"
).reset_index()
coal_pivot.columns = ["facility_id", "year", "coal_clean_tonnes", "coal_mined_tonnes"]

# Minerals - pivot by type (ore mined vs ore processed)
minerals_pivot = clean_minerals.pivot_table(
    index=["facility_id", "year"],
    columns="type",
    values="value_tonnes",
    aggfunc="sum"
).reset_index()
minerals_pivot.columns = ["facility_id", "year", "ore_mined_tonnes", "ore_processed_tonnes"]

#  Commodities - aggregate sum per facility_id + year
commodities_pivot = clean_commodities.groupby(["facility_id", "year"]).agg({
    "value_tonnes": "sum",
    "grade_ppm": "mean",
    "recovery_rate": "mean"
}).reset_index()
commodities_pivot.columns = ["facility_id", "year", "commodity_tonnes", "avg_grade_ppm", "avg_recovery_rate"]

#  Reserves - aggregate sum per facility_id + yearreserves_pivot = clean_reserves.groupby(["facility_id", "year"]).agg({
    "mineral_value_tonnes": "sum",
    "commodity_value_tonnes": "sum"
}).reset_index()
reserves_pivot.columns = ["facility_id", "year", "reserve_mineral_tonnes", "reserve_commodity_tonnes"]

# 6. Waste - aggregate sum per facility_id + year
waste_pivot = clean_waste.groupby(["facility_id", "year"]).agg({
    "value_tonnes": "sum",
    "total_material_tonnes": "sum"
}).reset_index()
waste_pivot.columns = ["facility_id", "year", "waste_tonnes", "waste_total_material_tonnes"]

merged = facilities.copy()

# Left join each fact table on facility_id
tables_to_merge = [
    ("capacity", capacity_pivot),
    ("coal", coal_pivot),
    ("minerals", minerals_pivot),
    ("commodities", commodities_pivot),
    ("reserves", reserves_pivot),
    ("waste", waste_pivot),
]

for name, df in tables_to_merge:
    # Merge on facility_id only (to keep all facilities)
    merged = merged.merge(df, on="facility_id", how="left")
    print(f"Merged {name}: {len(df)} rows → merged now has {len(merged.columns)} columns")

# Melt prices to long format for merging
prices_long = prices.melt(id_vars="Year", var_name="commodity", value_name="price")
prices_long.rename(columns={"Year": "year"}, inplace=True)

# Merge prices: match commodity to primary_commodity
merged = merged.merge(prices_long, on="year", how="left")

print(f"\nFinal merged dataset:")
print(f"  Rows: {len(merged)}")
print(f"  Columns: {len(merged.columns)}")
print(f"  Column names: {list(merged.columns)}")